# SafeMod - Jigsaw evaluation (Colab, GPU)
**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells in order.

Step 1 scores the 63,978 labelled test comments once with Detoxify `original` (about 5-15 min on a T4). Step 2 evaluates MAX, Mean, Noisy-OR and the proposed operator against the ground-truth labels (about 2-5 min, CPU).

In [ ]:
!nvidia-smi -L

## 1. Code
Upload `R3_code_and_results.zip` in the Files panel (left), then run:

In [ ]:
!unzip -o -q R3_code_and_results.zip -d /content/r3
%cd /content/r3/safemod
!pip install -q detoxify

## 2. Data: `test.csv` and `test_labels.csv`
First, while signed in to Kaggle, open https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/rules and accept the competition rules (one-time).

**Option A - Kaggle API (recommended):** on kaggle.com/settings, under *API*, click **Generate New Token** and copy the token it shows. Run the next cell and paste it when asked (it stays hidden).
If you instead have a legacy `kaggle.json` (Settings > *Create Legacy API Key*), see the comment inside the cell.

In [ ]:
!pip install -q --upgrade "kaggle>=1.8"
import os, getpass
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Paste your Kaggle API token: ')
# Legacy alternative: upload kaggle.json, then
# os.makedirs('/root/.kaggle', exist_ok=True); os.replace('kaggle.json', '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge -p data
!cd data && unzip -o -q jigsaw-toxic-comment-classification-challenge.zip && unzip -o -q test.csv.zip && unzip -o -q test_labels.csv.zip
!ls -la data

**Option B - manual:** download `test.csv.zip` and `test_labels.csv.zip` from the competition's *Data* tab, upload them to `/content/safemod_mc/data/` and run:
```
!cd data && unzip -o -q test.csv.zip && unzip -o -q test_labels.csv.zip
```

## 3. (Optional) keep progress on Google Drive
If Colab disconnects, re-running step 4 resumes where it stopped - but only if the output folder survives. Mount Drive and use the Drive path as `OUT`.

In [ ]:
OUT = 'jigsaw_outputs'
# from google.colab import drive; drive.mount('/content/drive'); OUT = '/content/drive/MyDrive/safemod_jigsaw'

## 4. Score the test set (GPU)

In [ ]:
!python jigsaw/score_jigsaw.py --data data --out {OUT} --batch-size 64

## 5. Evaluate (CPU)

In [ ]:
!python jigsaw/eval_jigsaw.py --pred {OUT}/jigsaw_predictions.csv

In [ ]:
import pandas as pd, json, hashlib, importlib.metadata
print('detoxify:', importlib.metadata.version('detoxify'))
h = hashlib.sha256(open(f'{OUT}/jigsaw_predictions.csv','rb').read()).hexdigest()
print('predictions sha256:', h)
print('identical to 18 Nov run:', h == '0e9522cc56e283ed24c56dd82957605fcea7a1a1ea8038ba8a7c236941512a71')
print(json.dumps(json.load(open(f'{OUT}/scoring_manifest.json')), indent=1)[:1500])
print(pd.read_csv(f'{OUT}/eval/checks.csv').to_string(index=False))

## 6. Download everything

In [ ]:
!zip -r -q jigsaw_outputs.zip {OUT}
from google.colab import files; files.download('jigsaw_outputs.zip')